PREDICTING STARTUP SURVIVAL: A MACHINE LEARNING PERSPECTIVE
 
> In questa tesi vengono combinate tecniche di Data Mining e Machine Learning
con l’obiettivo di identificare i principali fattori di rischio per le startup (con un
particolare focus su Competitors, Team e Funding), in modo da prevederne l’eventuale fallimento.

> I dati presi in esame provengono dalla piattaforma PitchBook, e
sono stati elaborati in linea con la letteratura economica recente.

> A partire dalle metriche calcolate, sono stati addestrati diversi modelli (Alberi
Decisionali, Random Forest, Reti Neurali Artificiali), i quali sono stati successivamente valutati in base alla capacità di classificare correttamente le startup fallite.
Questi esperimenti hanno permesso, inoltre, di cogliere i principali indicatori di "sopravvivenza" delle startup, mediante il valore dell’importanza che ogni modello
assegna a una determinata feature. Un ulteriore studio è stato svolto tramite l’impiego del Random Survival Forest, una versione alternativa del Random Forest
ideata per l’analisi della sopravvivenza.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight

import yaml

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from src.preprocessing import processUniversityList, getFlagTop50Institute,lump_categories

import matplotlib.pyplot as plt # Necessario per salvare i grafici
import seaborn as sns


from src.models.MLP import MLP

import scipy.integrate
if not hasattr(np, 'trapz'):
    np.trapz = scipy.integrate.trapezoid

with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)


Lettura della versione iniziale del dataset "master" 

In [ ]:
df = pl.read_csv(config['paths']['raw_dataset'],null_values=["NA"])

# Sostituisco "Stay" con il valore attuale di "GrowthStageGroup"
df = df.with_columns(
    pl.when(pl.col("GrowthNextStageGroup") == "Stay")
    .then(pl.col("GrowthStageGroup"))
    .otherwise(pl.col("GrowthNextStageGroup"))
    .alias("GrowthNextStageGroup")
)

Pre-processing
Il dataset di partenza è un panel in cui ogni azienda presenta un record per ogni anno di vita disponibile.
Nell' ottica di sperimentare con modelli di classificazione, si è scelto di focalizzarsi sui dati delle aziende che si trovano in fase di Early per provare a prevederne la condizione al qT-esimo anno successivo. Per fare ciò, il dataset viene raggruppato per azienda, mantenendo i valori delle features all'entrata nella fase Early; la variabile target viene invece mantenuta al T-esimo anno successivo o all'ultimo dato disponibile.   

Per evitare troppa diversità tra le aziende sono stati applicati alcuni filtri:
 - Il valore di Age all'entrata della fase Early deve essere <=2 (è così per la maggior parte delle aziende a parte alcuni outliers che vengono così eliminati)
 - L'anno di fondazione deve essere compreso tra il 2010 e il 2024-T (i dati si fermano al 2024)


In [ ]:
T=5

lastYear=2024-T

df_preseed = df.filter(
    pl.col("GrowthStageGroup")
    .eq("Early")
    .any()
    .over("CompanyID")
)
df_preseed=df_preseed.drop_nulls(subset=['GrowthStageGroup'])


df_preseed = df_preseed.select(["CompanyID", "Age"])

# aggregazioni per azienda
df_combined = (
    df_preseed
    .group_by("CompanyID")
    .agg([
        pl.col("Age").min().alias("StartingAge"),
        pl.col("Age").max().alias("LastAge"),
    ])
    .with_columns(
        pl.min_horizontal(
            pl.col("StartingAge") + T,
            pl.col("LastAge")
        ).alias("TargetAge")
    )
    .filter(pl.col("StartingAge") <= 2) # filtro aggiunto per evitare valori troppo diversi (in ogni caso la maggior parte delle aziende aveva un Age compreso tra 0 e 2 al momento dell'entrata in preseed)
)

df = df.join(df_combined, on="CompanyID")


df_target = df.filter(
    pl.col("Age") == pl.col("TargetAge")
).select(["CompanyID", "StartingAge" ,"TargetAge" ,'GrowthStageGroup','TimeNextStageGroup','GrowthNextStageGroup'])


df_target = df_target.with_columns(
    pl.when(pl.col("TargetAge") + pl.col("TimeNextStageGroup")<=pl.col("StartingAge") + 5) 
    .then(pl.col("GrowthNextStageGroup"))
    .otherwise(pl.col("GrowthStageGroup"))
    .alias("Target")
)

df_target_panel=df.join(df_target.select(["CompanyID", "Target"]),on='CompanyID')


df_target_final = df_target_panel.filter(
    (pl.col("Age") == pl.col("StartingAge")) &
    (pl.col("YearFounded") <= lastYear) &
    (pl.col("YearFounded") >= 2010)
)




Feature selection

In [ ]:
df_target_final=df_target_final.select(['CompanyID', 'Target' ,'YearFounded', 'Year_Delta','Age', 'N_Deal', 'TotalRaised_Est', 
                                        'Percent_Females', 'Is_Eco', 'Is_Eng', 'Is_NS', 
                                        'Is_Hum', 'Is_SS', 'Is_Med', 'Is_Law', 'Is_IT', 
                                         'Institute', 'WorkExp_Idx_Mean', 'Total_Founders', 'Is_Debt', 
                                         'Is_SpinOff', 'Is_CrowdFunding',  
                                        'MeanMedianRoundAmount_cum', 'Is_Accelerator', 'has_Corporate', 
                                        'has_VentureCapital', 'has_PublicInvestor', 'has_Angel_Lead', 'has_Corporate_Lead',
                                        'has_VentureCapital_Lead', 'has_Accelerator_Lead', 'has_PrivateEquity_Lead', 'has_PublicInvestor_Lead', 
                                          'HQCountry', 'PrimaryIndustrySector',
                                        'SimilarityScoreMean', 'N_Competitors', 'Same_Country','Highest_Degree_CEO','Gender_CEO','MeanTotalInvestments_cum',
                                        'WorkExperienceIndex_CEO', 'Is_Angel',  'Total_People', 'Is_Grant','has_PrivateEquity','TotalInvestors', 'Highest_Degree_Mean','Avg_Earliest_Year'
                                        ])
                                       
 # 'MeanTotalActivePortfolio_cum'


Viene creata la feature booleana HasTop50Institute che ha valore positivo se almeno una delle Università citate nella colonna 'Institutes' è presente nella classifica delle 50 migliori università al mondo. La classifica è stata stilata da QS World University Rankings ed è contenuta nel file QS_World_Rankings.csv.

Per fare ciò è stato svolto un confronto di stringhe tramite Jaccard Similarity.

In fase di preprocessing e di match sono state utilizzate diverse tecniche di NLP:

> Text Normalization: 
    Prima del confronto, il codice "pulisce" il testo per ridurre la variabilità semantica. Tutto il testo viene convertito in minuscolo. Vengono rimossi caratteri speciali come virgole, trattini e spazi multipli tramite Espressioni Regolari. Il codice rimuove parole comuni che non aiutano a distinguere un'università dall'altra, come "university", "of", "the". In NLP, queste sono considerate "stop words" specifiche del dominio accademico.

> Nella funzione calculate_similarity_by_words, il codice trasforma le stringhe in insiemi di parole: La stringa viene spezzata in singole unità (parole) usando lo spazio come delimitatore (Tokenizzazione). Trasformando la lista in un set(), il codice ignora l'ordine delle parole e le ripetizioni, concentrandosi solo sulla presenza dei termini (Bag of Words).

> Viene implementata una logica di Named Entity Disambiguation basata sugli acronimi: Tramite la Regex r"\((.*?)\)" viene estratto il testo tra parentesi (es. "MIT" da "Massachusetts Institute of Technology (MIT)"). Il confronto avviene su due binari: se l'acronimo combacia, l'istituto è confermato immediatamente, riducendo i falsi negativi dovuti a nomi troppo lunghi o complessi.

> Logica di Matching Ibrida: Il confronto finale non è una semplice uguaglianza, ma un sistema a cascata: Exact Match su Acronimo: Se l'acronimo estratto è presente nella lista Top 50. Exact Match su Stringa Pulita: Se la stringa normalizzata è identica. Substring Matching: Verifica se un acronimo è contenuto all'interno del nome (es: "MIT" in "MIT Boston"). Fuzzy Matching (Soglia): Se la similarità di Jaccard supera il threshold=0.5.

In [ ]:
# Caricamento e pulizia università
top_50 = processUniversityList(config['paths']['raw_university_ranking'])

df_target_final=df_target_final.with_columns(pl.col("Institute").map_elements( lambda x: getFlagTop50Institute(x, top_50),return_dtype=pl.Boolean, 
        skip_nulls=False).alias("HasTop50Institute"))

df_target_final=df_target_final.drop("Institute")

counts = df_target_final["HasTop50Institute"].value_counts()



Rimozione nazioni non abbastanza frequenti e frequency encoding della colonna 'HQCountry'

Frequency encoding colonna 'PrimaryIndustryGroup'

In [ ]:
df_target_final = lump_categories(df_target_final, "HQCountry", min_count=1000)


#frequency encoding HQCountry
freq_df = df_target_final.group_by("HQCountry").agg(
    pl.len().alias("HQCountryFreq")
)

df_target_final = df_target_final.join(freq_df, on="HQCountry").drop("HQCountry")


#frequency encoding PrimaryIndustrySector

freq_df = df_target_final.group_by("PrimaryIndustrySector").agg(
    pl.len().alias("PrimaryIndustrySectorFreq")
)

df_target_final = df_target_final.join(freq_df, on="PrimaryIndustrySector").drop("PrimaryIndustrySector")


# feature indicatore per il genere del CEO
df_target_final = df_target_final.to_dummies("Gender_CEO").drop("Gender_CEO_Male")



Gestione Missing Values

Scarto le righe che hanno missing valueas per Total_People, in quanto presentano lo stesso problema per la maggior parte delle features del team

In [ ]:
df_target_final = df_target_final.drop_nulls(subset=['Total_People'])


In [ ]:
df_target_final = df_target_final.with_columns(
    pl.col(pl.Boolean).cast(pl.Int8)
)

# #Per queste variabili si è deciso di imputare i missing values a 0
variabili_toInt = [ 'N_Competitors','Same_Country'] 

df_target_final = df_target_final.with_columns(
    [pl.col(name).cast(pl.Int64).fill_null(0) for name in variabili_toInt]

)

df_target_final = df_target_final.with_columns(
    pl.col('SimilarityScoreMean').fill_null(pl.col('SimilarityScoreMean').mean()) 

)

#Conversione mantenendo i missing values 
df_target_final = df_target_final.with_columns(
    pl.col(["YearFounded","Age",'Total_Founders','Total_People','Is_Eco','Is_Eng','Is_NS','Is_Hum','Is_SS','Is_Med','Is_Law','Is_IT']).cast(pl.Int64)
)


Missing values per feature

In [ ]:
# 1. Definiamo la soglia (40%)
threshold = 0.4

# 2. Identifichiamo le colonne da tenere
# Calcoliamo la frazione di null per ogni colonna e filtriamo i nomi
cols_to_keep = [
    col for col in df_target_final.columns 
    if df_target_final[col].null_count() / len(df_target_final) < threshold
]

# 3. Sovrascriviamo il dataframe (o creiamone uno nuovo)
df_target_final = df_target_final.select(cols_to_keep)

Conversione delle variabili booleane in numeri interi, revisione tipi variabili numeriche.

In [ ]:
result = (
    df_target_final.select([
        (pl.col(col).null_count() / pl.len() * 100).alias(col)
        for col in df_target_final.columns
    ])
    .unpivot(variable_name="colonna", value_name="percentuale_missing")
    .filter(pl.col("percentuale_missing") > 0)
    .sort("percentuale_missing", descending=True)
)

print(result)

Missing values per i record

In [ ]:
# 1. Conta i null in modo orizzontale (molto più veloce e sicuro)
df_with_row_counts = df_target_final.with_columns(
    null_count_row = pl.sum_horizontal(pl.all().is_null())
)

# 2. Raggruppa per vedere la distribuzione
distribuzione_missing = (
    df_with_row_counts
    .group_by("null_count_row")
    .agg(pl.len().alias("numero_di_righe"))
    .sort("null_count_row")
)

print(distribuzione_missing)

In [ ]:
iniziale=len(df_target_final)
# Drop delle righe con 4 o più missing values
print(f"Dataset prima del drop: {iniziale} righe")

df_target_final = df_with_row_counts.filter(pl.col("null_count_row") < 4).drop("null_count_row")

finale=len(df_target_final)
print(f"Righe rimosse: {iniziale - finale}")


Grafico bilanciamento Target

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Calcolo delle frequenze direttamente dal dataset
target_counts = df_target_final['Target'].value_counts()

# Estraiamo etichette e valori in ordine
labels = target_counts["Target"].to_list()
sizes = target_counts["count"].to_list()

# 2. Definizione colori (Palette 'Set1' di Matplotlib, molto simile alla tua immagine)
# Se hai più di 4 categorie, questa palette si adatta automaticamente
colors = plt.get_cmap('Set1').colors

# 3. Creazione del grafico
fig, ax = plt.subplots(figsize=(8, 8))

# autopct calcola la percentuale reale dai dati
patches, texts, autotexts = ax.pie(
    sizes, 
    labels=labels, 
    autopct='%1.1f%%', 
    colors=colors, 
    startangle=140,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
    textprops={'fontsize': 11}
)

# Rendiamo le percentuali interne più leggibili (bianche o nere a seconda del gusto)
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_weight('bold')

# 4. Estetica finale
plt.axis('equal') 

plt.tight_layout()
plt.show()

Creazione dataset per la classificazione binaria

In [ ]:
dataset = df_target_final.with_columns(
    pl.when(pl.col("Target").is_in(["Out"]))
    .then(1)
    .otherwise(0)
    .alias("Target") # Dai un nome alla nuova colonna o usa "Target" per sovrascrivere
)

dataset.write_csv(config['paths']['dataset'])


Grafici sulla distribuzione dei valori delle feature

In [ ]:
import matplotlib.pyplot as plt
import math
import pandas as pd

# Definizione dei gruppi (aggiornati secondo la tua ultima versione)
context_company = [
     'Age', 'YearFounded'
]

context_location =  ['HQCountry']

context_industry=['PrimaryIndustrySector']

context_competitors=['N_Competitors', 'SimilarityScoreMean',  'Same_Country',]



context_founders = [
    'Total_Founders', 'Percent_Females',  'Highest_Degree_Mean', 'WorkExp_Idx_Mean', 'HasTop50Institute',
    'Is_Eco', 'Is_Eng', 'Is_NS', 'Is_Hum', 'Is_SS', 'Is_Med', 'Is_Law', 'Is_IT'    
]

context_employees= ['Total_People', 'Gender_CEO_Female', 'Gender_CEO_null', 'Highest_Degree_CEO', 'WorkExperienceIndex_CEO']

context_funding = [
    'N_Deal', 'TotalRaised_Est', 'MeanMedianRoundAmount_cum','MeanTotalInvestments_cum', 'Is_Accelerator', 
     'Is_Angel',  'Is_Debt', 'Is_SpinOff', 'Is_CrowdFunding','Is_Grant',  'Avg_Earliest_Year'
]
context_investors = ['TotalInvestors', 'has_PrivateEquity','has_Corporate', 'has_VentureCapital', 'has_PublicInvestor', 'has_Angel_Lead', 
    'has_Corporate_Lead','has_VentureCapital_Lead', 'has_Accelerator_Lead', 
    'has_PrivateEquity_Lead', 'has_PublicInvestor_Lead']

import matplotlib.pyplot as plt
import math
import pandas as pd

def plot_topic_distributions(df, features, title_group):
    feat_numeric = [f for f in features if f in df.columns and df[f].dtype != 'object' and df[f].dtype != 'string']
    feat_categorical = [f for f in features if f in df.columns and (df[f].dtype == 'object' or df[f].dtype == 'string')]

    # --- PARTE 1: NUMERICHE ---
    if feat_numeric:
        n_cols_num = 6
        n_rows_num = math.ceil(len(feat_numeric) / n_cols_num)
        fig, axes = plt.subplots(n_rows_num, n_cols_num, figsize=(20, 3 * n_rows_num))
        if n_rows_num == 1 and n_cols_num == 1: axes = [axes]
        else: axes = axes.flatten()

        for i, col_name in enumerate(feat_numeric):
            data = df[col_name].dropna()
            unique_vals = sorted(data.unique())
            if len(unique_vals) <= 2:
                axes[i].hist(data, bins=[-0.5, 0.5, 1.5], edgecolor='black', alpha=0.8, rwidth=0.7, color='#3498db')
                axes[i].set_xticks([0, 1])
            else:
                axes[i].hist(data, bins=min(20, len(unique_vals)), edgecolor='black', alpha=0.8, rwidth=0.8, color='#3498db')
            axes[i].set_title(col_name, fontsize=10, fontweight='bold')

        for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
        plt.tight_layout()
        
        # SALVATAGGIO: Deve essere prima di show() e fuori dal loop
        plt.savefig(f"img/{title_group}_numeric.png", dpi=300, bbox_inches='tight')
        plt.show()

    # --- PARTE 2: CATEGORICHE ---
    if feat_categorical:
        n_cols_cat = 2
        n_rows_cat = math.ceil(len(feat_categorical) / n_cols_cat)
        fig, axes = plt.subplots(n_rows_cat, n_cols_cat, figsize=(20, 5 * n_rows_cat))
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

        for i, col_name in enumerate(feat_categorical):
            data = df[col_name].dropna()
            data = data[data != "Other"]
            top_cats = data.value_counts().head(10)
            axes[i].barh(top_cats.index.astype(str), top_cats.values, color='#e67e22', edgecolor='black')
            axes[i].invert_yaxis()
            axes[i].set_title(f"{col_name} (Top 10)", fontsize=12, fontweight='bold')

        for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
        plt.tight_layout()
        
        # SALVATAGGIO
        plt.savefig(f"img/{title_group}_categorical.png", dpi=300, bbox_inches='tight')
        plt.show()

# Esempio di utilizzo
# plot_topic_distributions(dataset, context_company, "Context Company")
dataset = pd.read_csv(config["paths"]["dataset"])

# Esempio di utilizzo:
plot_topic_distributions(dataset, context_industry,'Industry')


Correlazione features

In [ ]:
dataset = pd.read_csv(config["paths"]["dataset"])
corr_matrix = dataset.drop(['CompanyID', 'Target'],axis=1).corr()


# plt.figure(figsize=(30,20))
# ax = sns.heatmap(data = corr_matrix,cmap='YlGnBu',annot=True)

# bottom, top = ax.get_ylim()
# ax.set_ylim(bottom + 0.5,top - 0.5)


# Trasforma la matrice in una serie di coppie (stack)
# e rimuove i duplicati (unstack della parte superiore della matrice)
sol = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                  .stack()
                  .reset_index())

sol.columns = ['Variabile_1', 'Variabile_2', 'Correlazione']

# Applica il filtro per i range richiesti
mask = (
    ((sol['Correlazione'] >= 0.90) & (sol['Correlazione'] <= 0.99)) |
    ((sol['Correlazione'] <= -0.90) & (sol['Correlazione'] >= -0.99))
)

risultato = sol[mask].sort_values(by='Correlazione', ascending=False)

print(risultato)





In [ ]:
import wandb
import xgboost as xgb

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import shap
import matplotlib.pyplot as plt # Necessario per salvare i grafici


dataset = pd.read_csv(config["paths"]["dataset"])

entity="giuliocasti-universit-degli-studi-di-cagliari"
project="startup-prediction-exp2"
# Inizializza lo sweep
sweep_id = wandb.sweep(config["sweep_settings"],entity=entity, project=project)




In [ ]:
ids = dataset['CompanyID']  # Salva la colonna ID
X = dataset.drop(['CompanyID', 'Target'],axis=1)
y = dataset['Target']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config['test_size'], stratify=y,random_state=config['random_seed'])


imputer = SimpleImputer(strategy='median')#mean

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)



scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

def train():
    with wandb.init():
        wandb_config = wandb.config
        
        # --- RANDOM FOREST ---
        if wandb_config.model_type == "rf":
            model = RandomForestClassifier(
                n_estimators=wandb_config.n_estimators,
                max_depth=wandb_config.max_depth, 
                min_samples_leaf=wandb_config.min_samples_leaf,
                max_features='log2',
                # min_samples_split=wandb_config.min_samples_split,
                # BILANCIAMENTO: Pesa le classi in base alla frequenza
                class_weight='balanced',
                random_state=config['random_seed']
            )

            model.fit(X_train, y_train)
            preds = model.predict(X_test)  # Usa dati imputed ma non scaled per RF
            preds_train=model.predict(X_train)

        # --- XGBOOST ---
        elif wandb_config.model_type == "xgb":
            # Calcolo del rapporto per scale_pos_weight (80/20 -> 4)
            ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])
            
            model = xgb.XGBClassifier(
                n_estimators=wandb_config.n_estimators, 
                learning_rate=wandb_config.learning_rate,
                max_depth=wandb_config.max_depth,
                # BILANCIAMENTO: Penalizza l'errore sulla classe minoritaria
                scale_pos_weight=ratio, 
                min_child_weight=wandb_config.min_child_weight, 
                subsample=wandb_config.subsample,               
                colsample_bytree=wandb_config.colsample_bytree, 
                random_state=config['random_seed']
            )

            model.fit(X_train, y_train)
            preds = model.predict(X_test)  # Usa dati imputed ma non scaled per XGB
            preds_train=model.predict(X_train)

        elif wandb_config.model_type == "mlp":
            # BILANCIAMENTO: Oversampling specifico per MLP
            # Scegliamo SMOTE per creare nuovi punti e aiutare la rete a generalizzare
            

            model = MLPClassifier(
                hidden_layer_sizes=wandb_config.hidden_layer_sizes,
                activation='relu',
                solver='adam',
                alpha=wandb_config.alpha,
                learning_rate_init=wandb_config.learning_rate_init,
                random_state=config['random_seed'],
                max_iter=500,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10
            )
           
            # Usiamo i dati ricampionati (X_res, y_res)
            model.fit(X_res, y_res)
            preds = model.predict(X_test_scaled)
            preds_train=model.predict(X_train_scaled)

     
                
        # Training e Score
        acc = accuracy_score(y_test, preds)
        acc_train=accuracy_score(y_train, preds_train)

        precision = precision_score(y_test, preds, zero_division=0)
        recall = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        f1_train = f1_score(y_train, preds_train, zero_division=0)
        
        # Log delle metriche con F1 come prioritario
        wandb.log({
            "accuracy_train": acc_train,
            "F1_train": f1_train,
            "accuracy": acc,
            "F1": f1,
            "precision": precision,
            "recall": recall,
        })

        # wandb.sklearn.plot_confusion_matrix(y_test, preds, ["Neg", "Pos"])
        if(False):
            # Selezione del campione per SHAP
            explainer_sample = X_test_scaled[:100] 
            
            if wandb_config.model_type in ["rf", "xgb"]:
                explainer = shap.TreeExplainer(model)
                shap_result = explainer.shap_values(explainer_sample)
                
                # Gestione differenze tra RF e XGBoost
                if isinstance(shap_result, list):
                    # Solitamente RF restituisce una lista [classe_0, classe_1]
                    shap_values = shap_result[1]
                else:
                    # XGBoost restituisce spesso direttamente i valori per la classe positiva
                    shap_values = shap_result
            else:
                # --- MLP / Kernel SHAP ---
                # Ridurre il background a 20-30 campioni per evitare tempi biblici
                background = shap.sample(X_res, 30) 
                explainer = shap.KernelExplainer(model.predict_proba, background)
                
                # n_samples controlla il numero di perturbazioni (più alto = più preciso ma lento)
                shap_values_output = explainer.shap_values(explainer_sample, n_samples=100)
                
                # Estrazione forzata per evitare l'AssertionError
                if isinstance(shap_values_output, list):
                    # Prendiamo la classe 1 (Out of Business)
                    shap_values = shap_values_output[1]
                else:
                    # Se SHAP restituisce un array 3D: (campioni, feature, classi)
                    if len(shap_values_output.shape) == 3:
                        shap_values = shap_values_output[:, :, 1]
                    else:
                        shap_values = shap_values_output

            # --- CREAZIONE E LOG DEL GRAFICO ---
            plt.figure(figsize=(10, 6))
            
            # Generiamo il plot senza mostrarlo subito (show=False)
            shap.summary_plot(
                shap_values, 
                explainer_sample, 
                feature_names=X.columns.tolist(), 
                show=False
            )
            
            # Log su WandB come immagine
            wandb.log({"shap_summary_plot": wandb.Image(plt)})

        # perm_importance = permutation_importance(model, X_test_scaled, y_test, scoring='accuracy', random_state=config['random_seed'])


        # sorted_importances_idx = perm_importance.importances_mean.argsort()[::-1]
        
        # # top_30_names = [X.columns[i] for i in sorted_importances_idx[:30]]
        


        # # # 2. Log come stringa unica su W&B (così appare nei grafici e nei log)
        # # top_30_str = "['"+"', '".join(top_30_names)+"']"
        # # wandb.log({"top_features_text": top_30_str})

        # importances = pd.DataFrame(
        #     perm_importance.importances[sorted_importances_idx].T,
        #     columns=X.columns[sorted_importances_idx],
        # )

        #     # Calcola l'altezza dinamica in base al numero di feature
        # fig_height = len(X.columns) * 0.3 
        # fig, ax = plt.subplots(figsize=(10, fig_height))

        # importances.plot.box(vert=False, whis=10, ax=ax)
        # ax.set_title("Permutation Importances (test set)")
        # ax.axvline(x=0, color="r", linestyle="--", alpha=0.5)
        # ax.set_xlabel("Diminuzione della performance")
        # plt.tight_layout()

        # # Log su W&B
        # wandb.log({"permutation_importance_plot": wandb.Image(plt)})
        # plt.close(fig) # Chiudi la figura per liberare memoria


In [ ]:
wandb.agent(sweep_id, function=train, count=20)